<a href="https://colab.research.google.com/github/NxrFesdac/bourbaki-nlp-avanzado/blob/main/modulo4/NED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install wikipedia scikit-learn

In [2]:
!python -m spacy download es_core_news_lg

^C


In [ ]:
# =====================================================================
# EJERCICIO 1: Desambiguación Simple Basada en Diccionario (N-gramas)
# =====================================================================
print("="*60)
print("EJERCICIO 1: Desambiguación con N-gramas y Diccionario")
print("="*60)

# 1. Base de Conocimiento
base_conocimiento = {
    "ID_001": {
        "nombre": "Mercurio",
        "descripcion": "Es el planeta del sistema solar más próximo al Sol y el más pequeño. Tiene muchos cráteres debido al impacto de meteoritos."
    },
    "ID_002": {
        "nombre": "Mercurio",
        "descripcion": "Es un elemento químico, un metal pesado plateado que es líquido a temperatura ambiente y se usaba en antiguos termómetros."
    }
}

# 2. La Entrada (Nuestro ejemplo común para todos los ejercicios)
oracion = "La sonda espacial estudió Mercurio, el planeta más pequeño del sistema solar, y descubrió nuevos cráteres."
mencion = "Mercurio"
contexto = set(oracion.lower().split())

# 3. Lógica Simple de Desambiguación
puntuaciones = {}
for id_entidad, datos in base_conocimiento.items():
    # Contar cuántas palabras de la descripción coinciden con nuestra oración
    palabras_descripcion = set(datos["descripcion"].lower().split())
    interseccion = len(contexto.intersection(palabras_descripcion))
    puntuaciones[id_entidad] = interseccion

# 4. Enlace Final
mejor_coincidencia = max(puntuaciones, key=puntuaciones.get)
print(f"Contexto: '{oracion}'")
print(f"Mención: '{mencion}'")
print(f"-> Enlazado '{mencion}' a {mejor_coincidencia}: {base_conocimiento[mejor_coincidencia]['descripcion']}\n")

EJERCICIO 1: Desambiguación con N-gramas y Diccionario
Contexto: 'La sonda espacial estudió Mercurio, el planeta más pequeño del sistema solar, y descubrió nuevos cráteres.'
Mención: 'Mercurio'
-> Enlazado 'Mercurio' a ID_001: Es el planeta del sistema solar más próximo al Sol y el más pequeño. Tiene muchos cráteres debido al impacto de meteoritos.



In [ ]:
# =====================================================================
# EJERCICIO 2: Desambiguación usando Wikipedia (TF-IDF y Coseno)
# =====================================================================
print("="*60)
print("EJERCICIO 2: Desambiguación Simple con Wikipedia")
print("="*60)

import wikipedia
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Configurar Wikipedia en español
wikipedia.set_lang("es")

# 2. Los Datos
# Usamos el mismo ejemplo, e incluimos otro caso para demostrar la diferencia.
conjunto_datos = [
    {
        "mencion": "Mercurio",
        "contexto": "La sonda espacial estudió Mercurio, el planeta más pequeño del sistema solar, y descubrió nuevos cráteres.",
    },
    {
        "mencion": "Mercurio",
        "contexto": "El termómetro antiguo se rompió y derramó el mercurio líquido en el suelo del laboratorio.",
    }
]

def desambiguar_wikipedia(mencion, contexto):
    """Enlaza una mención a la página correcta de Wikipedia usando TF-IDF."""

    import warnings
    warnings.filterwarnings("ignore", category=UserWarning, module='wikipedia')

    # Paso 1: Buscar en Wikipedia los 5 mejores títulos candidatos
    try:
        candidatos = wikipedia.search(mencion, results=5)
    except Exception as e:
        return "Error en la búsqueda", 0.0

    documentos = [contexto]       # El primer documento es nuestro contexto
    candidatos_validos = []       # Mantener un registro de los candidatos sin errores

    # Paso 2: Obtener las primeras 2 oraciones de la página de Wikipedia de cada candidato
    for candidato in candidatos:
        try:
            resumen = wikipedia.summary(candidato, sentences=2, auto_suggest=False)
            documentos.append(resumen)
            candidatos_validos.append(candidato)
        except Exception:
            # Omitir candidatos que lanzan errores de Desambiguación o Página
            continue

    if not candidatos_validos:
        return "No se encontraron coincidencias", 0.0

    # Paso 3: Convertir texto en vectores y calcular la Similitud del Coseno
    # documentos = [Contexto, Resumen 1, Resumen 2, ...]
    palabras_vacias = ["el", "la", "los", "las", "un", "una", "unos", "unas",
                       "de", "del", "a", "al", "en", "por", "para", "con",
                       "y", "o", "su", "sus", "se", "es", "fue", "que", "como"]
    vectores = TfidfVectorizer(stop_words=palabras_vacias).fit_transform(documentos)

    # Comparar el contexto (índice 0) contra todos los resúmenes (índice 1 en adelante)
    puntuaciones = cosine_similarity(vectores[0:1], vectores[1:]).flatten()

    # Obtener el índice de la puntuación más alta y devolver el candidato correspondiente
    mejor_indice = puntuaciones.argmax()
    return candidatos_validos[mejor_indice], puntuaciones[mejor_indice]

for item in conjunto_datos:
    menc = item["mencion"]
    cont = item["contexto"]

    print(f"Mención: '{menc}'")
    print(f"Contexto: '{cont}'")

    mejor_entidad, puntuacion = desambiguar_wikipedia(menc, cont)

    print(f"-> Entidad Predicha en Wikipedia: {mejor_entidad}")
    print(f"-> Puntuación de Confianza: {puntuacion:.4f}\n")


EJERCICIO 2: Desambiguación Simple con Wikipedia
Mención: 'Mercurio'
Contexto: 'La sonda espacial estudió Mercurio, el planeta más pequeño del sistema solar, y descubrió nuevos cráteres.'
-> Entidad Predicha en Wikipedia: Mercurio (planeta)
-> Puntuación de Confianza: 0.3451

Mención: 'Mercurio'
Contexto: 'El termómetro antiguo se rompió y derramó el mercurio líquido en el suelo del laboratorio.'
-> Entidad Predicha en Wikipedia: Mercurio (elemento)
-> Puntuación de Confianza: 0.1264



In [7]:
import wikipedia

wikipedia.set_lang("es")
wikipedia.search("Mercurio", results=5)
wikipedia.summary("Mercurio (planeta)", sentences=2, auto_suggest=False)

'Mercurio es el planeta del sistema solar más cercano al Sol y el más pequeño. Recibió su nombre del hijo del dios romano Júpiter seg.'

In [ ]:
# =====================================================================
# EJERCICIO 3: Pipeline Completo (NER con spaCy + NED con Wikipedia)
# =====================================================================
print("="*60)
print("EJERCICIO 3: Pipeline Completo (Extracción NER + Desambiguación)")
print("="*60)

import spacy

# 1. Cargar el modelo NER de spaCy (modelo pequeño en español)
print("Cargando el modelo spaCy 'es_core_news_sm'...")
nlp = spacy.load("es_core_news_lg")

# 2. Los Datos (en español) - Usamos la misma oración original
texto_ejemplo = "La sonda espacial estudió Mercurio, el planeta más pequeño del sistema solar, y descubrió nuevos cráteres."

# --- Fase 1: Reconocimiento de Entidades Nombradas (NER) ---
def extraer_entidades(texto):
    """Extrae entidades nombradas del texto usando spaCy."""
    if nlp is None:
        return ["Mercurio"] # Fallback en caso de no tener el modelo

    doc = nlp(texto)

    # Solo nos interesan Personas (PER), Organizaciones (ORG) y Lugares (LOC)
    etiquetas_validas = ["PER", "ORG", "LOC"]
    entidades = [ent.text for ent in doc.ents if ent.label_ in etiquetas_validas]

    # Devolver entidades únicas para evitar desambiguar la misma palabra dos veces
    return list(set(entidades))


print(f"\nTexto de contexto: '{texto_ejemplo}'\n")

# Paso 1: Extraer (NER)
menciones_ner = extraer_entidades(texto_ejemplo)
print(f"🔍 Entidades Extraídas (NER): {menciones_ner}\n")

# Paso 2: Desambiguar (NED)
print("--- 🔗 Desambiguación (NED) ---")
for menc in menciones_ner:
    mejor_entidad, puntuacion = desambiguar_wikipedia(menc, texto_ejemplo)

    print(f"Mención: '{menc}'")
    print(f"  -> Entidad Predicha en Wikipedia: {mejor_entidad}")
    print(f"  -> Puntuación de Confianza: {puntuacion:.4f}\n")

EJERCICIO 3: Pipeline Completo (Extracción NER + Desambiguación)
Cargando el modelo spaCy 'es_core_news_sm'...

Texto de contexto: 'La sonda espacial estudió Mercurio, el planeta más pequeño del sistema solar, y descubrió nuevos cráteres.'

🔍 Entidades Extraídas (NER): ['Mercurio']

--- 🔗 Desambiguación (NED) ---
Mención: 'Mercurio'
  -> Entidad Predicha en Wikipedia: Mercurio (planeta)
  -> Puntuación de Confianza: 0.3451

